#### Note:

1. Формализовать текущий пайплайн в LangChain (Redis + ChromaDB);
2. Модифицировать его проверками на входе и выходе (необходимо избежать ситуации когда он отвечает абсруд);
3. Модифицировать его циклом на попытку ответа в случае неудачи.
4. Обязательно сделать его человеком при ответе пользователю (возможен routing на human-native answer);
5. Исследуем GPT-oss?

Help docs:

https://yandex.cloud/ru/docs/ai-studio/concepts/generation/models#sdk_1

# Import Libs

In [1]:
import json
from typing import TypedDict, List, Optional
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
import numpy as np
import asyncio
from abc import ABC

# Работаем с секретами
from dotenv import load_dotenv
import os

from functools import lru_cache
# Работаем с Redis \ ChromaDB

load_dotenv("../../.env")

from UnionChatBot.utils.ChatHistoryManager import ChatHistoryManager
from UnionChatBot.utils.ChromaAdapter import ChromaAdapter
from UnionChatBot.utils.RedisAdapters import SemanticRedisCache

# Define general args

In [2]:
openai_api_key=os.getenv("API_KEY")
FOLDER_ID = os.getenv("FOLDER_ID")
openai_api_base="https://llm.api.cloud.yandex.net/v1"
model=f"gpt://{FOLDER_ID}/gpt-oss-120b/latest"

In [3]:
class AgentState(TypedDict):
    embedding_input: np.ndarray
    user_id: str
    collection_name: str
    user_input: str
    model_output: str
    error: str
    is_valid_input: bool
    is_cached: bool
    is_valid_output: bool
    tasks: List[str]
    tasks_results: List[str]

# Define Two LLM Types

In [4]:
llm_yandex_light = ChatOpenAI(
    model=f"gpt://{FOLDER_ID}/yandexgpt-lite",
    temperature=0,
    openai_api_base=openai_api_base,
    openai_api_key=openai_api_key, 
)

In [5]:
llm_gpt = ChatOpenAI(
    model=f"gpt://{FOLDER_ID}/gpt-oss-120b/latest",
    temperature=0,
    openai_api_base=openai_api_base,
    openai_api_key=openai_api_key, 
)

# Define Embeder

In [6]:
import os
from typing import Any

import numpy as np
import requests
import time

from chromadb import Documents, EmbeddingFunction, Embeddings


class MyEmbeddingFunction(EmbeddingFunction):
    api_url = os.getenv(
        "EMBEDDING_API",
        "https://llm.api.cloud.yandex.net:443/foundationModels/v1/textEmbedding",
    )
    folder_id = os.getenv("FOLDER_ID", None)
    iam_token = os.getenv("API_KEY", None)
    time_sleep = float(os.getenv("TIME_SLEEP_RATE_EMBEDDER", 0.01))

    def __init__(
        self,
        doc_model_uri: str = None,
        query_model_uri: str = None,
        text_type: str = None,
        *args: Any,
        **kwargs: Any,
    ):
        """
        Initialize the embedding function with Yandex GPT API credentials.

        Args:
            doc_model_uri: Model URI for document embeddings
            query_model_uri: Model URI for query embeddings
            time_sleep: time between each query to yandex api.
        """
        super().__init__(*args, **kwargs)
        self.headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {self.iam_token}",
            "x-folder-id": self.folder_id,
        }
        # set default text type doc if not provided
        self.text_type = text_type or "doc"

        # Set default model URIs if not provided
        self.doc_model_uri = (
            doc_model_uri or f"emb://{self.folder_id}/text-search-doc/latest"
        )
        self.query_model_uri = (
            query_model_uri or f"emb://{self.folder_id}/text-search-query/latest"
        )

    @lru_cache(maxsize=128)
    def _get_single_embedding(self, text: str) -> np.ndarray:
        """
        Get embedding for a single text.

        Args:
            text: Input text to embed
            text_type: "doc" or "query"

        Returns:
            numpy.ndarray: Embedding vector
        """
        model_uri = (
            self.doc_model_uri if self.text_type == "doc" else self.query_model_uri
        )

        data = {"modelUri": model_uri, "text": text}
        time.sleep(self.time_sleep)
        response = requests.post(self.api_url, json=data, headers=self.headers)
        response.raise_for_status()

        return np.array(response.json()["embedding"])

    def __call__(self, input: Documents) -> Embeddings:
        if isinstance(input, str):
            input = [input]

        embeddings = []
        for text in input:
            embedding = self._get_single_embedding(text)
            embeddings.append(embedding)
        return np.array(embeddings)


# Define Main Agentic Logic

In [7]:
class AgentUtils(ABC):
    """ Стандартные возможности агента из коробки
    """
    @staticmethod
    def _load_prompt(path: str, how: str = "local") -> str:
        """ Читаем промпты для работы бота.

        Args:
            path: путь до файла;
            how: где хранится файл;

        Return:
            Текст промпта.
        """
        if how == "local":
            with open(path, "r", encoding="utf-8") as f:
                return f.read()
        elif how == "s3":
            with boto3.client('s3') as s3:
                bucket = s3_params["bucket"]
                key = s3_params["key"]
                obj = s3.get_object(Bucket=bucket, Key=key)
                return obj['Body'].read().decode('utf-8')
        else:
            raise NotImplementedError(f"Unknown source: {how}")
            
    def _get_user_history_text_sync(self, user_id: str) -> str:
        """Синхронно получает историю диалога пользователя"""
        try:
            history = self.chat_manager.get_user_history(user_id=user_id)
            return history if history else "История диалога отсутствует."
        except Exception as e:
            # logger.error(f"Error getting user history: {e}")
            return "История диалога недоступна."
    
    def _format_rag_documents(self, data: dict) -> str:
        """Форматирует документы из RAG в читаемый вид."""
        documents = data.get("documents", [])
        metadatas = data.get("metadatas", [])
        
        if not documents:
            return "<RAG>Документы для справки отсутствуют</RAG>"
        
        formatted_docs = []
        
        for idx, (doc_text, metadata) in enumerate(zip(documents, metadatas), 1):
            # Форматируем каждый документ
            formatted_doc = self._format_single_document(idx, doc_text, metadata)
            formatted_docs.append(formatted_doc)
        
        rag_content = "\n\n".join(formatted_docs)
        return rag_content
    
    def _format_single_document(self, index: int, doc_text: str, metadata: dict) -> str:
        """Форматирует один документ с метаданными."""
        # Очищаем и обрезаем текст документа
        clean_text = doc_text.strip()
        
        # Извлекаем информацию из метаданных
        source = metadata.get('source', 'Неизвестный источник')
        document_type = metadata.get('type', 'Документ')
        section = metadata.get('section', '')
        page = metadata.get('page', '')
        
        # Форматируем источник
        source_info = []
        if source and source != 'Неизвестный источник':
            source_info.append(f"Источник: {source}")
        if document_type and document_type != 'Документ':
            source_info.append(f"Тип: {document_type}")
        if section:
            source_info.append(f"Раздел: {section}")
        if page:
            source_info.append(f"Страница: {page}")
        
        source_line = " | ".join(source_info) if source_info else "Источник не указан"
        
        return f"""Документ #{index}:
    {clean_text}
    
    [{source_line}]"""

In [8]:
class ProfkomAgent(AgentUtils):
    invalid_user_input: str = "Ваш текст не прошел проверку фильтра и не может быть обработан."
    
    def __init__(self,
                 llm_easy,
                 llm_hard,
                 redis_cache: SemanticRedisCache,
                 chat_manager: ChatHistoryManager,
                 chroma_adapter: ChromaAdapter,
                 text_embeder: MyEmbeddingFunction,
                 prompt_dir="../../prompts",
                 s3_info: Optional[dict] = {}):
        self.llm_easy = llm_easy
        self.llm_hard = llm_hard
        self.redis_cache = redis_cache
        self.chat_manager = chat_manager
        self.chroma_adapter = chroma_adapter
        self.text_embeder = text_embeder
        
        self.s3_info = s3_info
        self.prompts = {
            "validate": self._load_prompt(f"{prompt_dir}/filter_prompt.txt", how="local"),
            "decompose": self._load_prompt(f"{prompt_dir}/decompose_question.txt", how="local"),
            "answer_part": self._load_prompt(f"{prompt_dir}/answer_part.txt", how="local"),
            "collect_final": self._load_prompt(f"{prompt_dir}/summary_results.txt", how="local"),
            "determine_collection": self._load_prompt(f"{prompt_dir}/determine_collection.txt", how="local")
        }

    async def check_redis_cache(self, state: AgentState) -> AgentState:
        """Проверяем кэш Redis (асинхронная версия)."""
        state["is_cached"] = False
        # Cоздаем ембединг для входящего запроса
        state["embedding_input"] = await asyncio.to_thread(self.text_embeder,
                                                           state["user_input"])
        # Асинхронно проверяем кэш Redis
        cached = await asyncio.to_thread(
            self.redis_cache.get,
            query_embedding=state["embedding_input"], 
            query=state["user_input"]
        )
        
        if cached:
            # Асинхронно сохраняем в историю
            await asyncio.to_thread(
                self.chat_manager.add_message_to_history,
                state["user_id"],
                cached["response"]
            )
            state["is_cached"] = True
            state["model_output"] = cached["response"]
        
        return state
        
    async def validate_text(self, state: AgentState) -> AgentState:
        """ Проверяем, что текст валидный исходя из логики промпта.

        Arg:
            state: Состояние агента на текущем шаге.

        Return:
            Обновленное состояние агента о том, что это валидный вопрос.
        """
        prompt = ChatPromptTemplate.from_template(self.prompts["validate"])
        chain = prompt | self.llm_easy
        response_obj = await chain.ainvoke({"text": state["user_input"]})
        response = response_obj.content.strip().lower() 

        if "valid" in response:
            state["is_valid_input"] = True
        else:
            state["is_valid_input"] = False
            state["model_output"] = self.invalid_user_input
            state["error"] = response
        return state

    async def _determine_collection_with_llm(self, text: str) -> str:
        """Использует LLM для определения коллекции на основе организации"""
        prompt = ChatPromptTemplate.from_template(self.prompts["determine_collection"])
        chain = prompt | self.llm_hard
        
        response_obj = await chain.ainvoke({"organization": text})
        response = response_obj.content.strip()
        
        # Парсим ответ LLM
        if "<КОЛЛЕКЦИЯ>" in response and "</КОЛЛЕКЦИЯ>" in response:
            start_idx = response.find("<КОЛЛЕКЦИЯ>") + len("<КОЛЛЕКЦИЯ>")
            end_idx = response.find("</КОЛЛЕКЦИЯ>")
            collection = response[start_idx:end_idx].strip()
        else:
            collection = response.strip()
        return collection

    async def decomposer(self, state: AgentState) -> AgentState:
        """Разделяем запрос пользователя на несколько вложенных с учетом истории."""
        # Получаем историю пользователя из базового класса
        user_history = await asyncio.to_thread(
        self._get_user_history_text_sync,  # ← передаем функцию, а не результат
        state["user_id"]                   # ← аргумент отдельно
        )
        
        prompt = ChatPromptTemplate.from_template(self.prompts["decompose"])
        chain = prompt | self.llm_hard

        user_question = state['user_input']
        
        response_obj = await chain.ainvoke({
            "user_question": user_question,
            "user_history": user_history})
        
        try:
            response_text = response_obj.content.strip()
            
            # Извлекаем задачи из блока <ЗАДАЧИ>
            if "<ЗАДАЧИ>" in response_text and "</ЗАДАЧИ>" in response_text:
                start_idx = response_text.find("<ЗАДАЧИ>") + len("<ЗАДАЧИ>")
                end_idx = response_text.find("</ЗАДАЧИ>")
                tasks_text = response_text[start_idx:end_idx].strip()
            else:
                tasks_text = response_text
            
            # Разделяем задачи по <PART>
            if "<PART>" in tasks_text:
                state["tasks"] = [task.strip() for task in tasks_text.split("<PART>") if task.strip()]
            else:
                state["tasks"] = [tasks_text]
                
        except Exception as e:
            state["error"] = f"Decomposition error: {str(e)}"
            state["tasks"] = [user_question]
        
        return state

    async def process_all_tasks(self, state: AgentState, max_concurrent: int = 3) -> AgentState:
        """Обрабатывает все подзадачи параллельно с ограничением количества одновременных запросов"""
        if not state.get("tasks"):
            state["error"] = "No tasks to process"
            state["tasks_results"] = []
            return state

        semaphore = asyncio.Semaphore(max_concurrent)
        prompt = ChatPromptTemplate.from_template(self.prompts["answer_part"])
        chain = prompt | self.llm_hard
        
        
        async def process_single_task(task: str) -> str:
            async with semaphore:
                documents = await asyncio.to_thread(
                    self.chroma_adapter.get_info,  # функция
                    task,                          # query
                    state["collection_name"]       # collection_name
                )
                doc_info = self._format_rag_documents(documents)
                response_obj = await chain.ainvoke({
                    "task_question": task,
                    "additional_documents": doc_info
                })
                return response_obj.content.strip()
        # Узнаем название коллекции исходя из вопроса пользователя
        state["collection_name"] = await self._determine_collection_with_llm(text=state["user_input"][:90])
        
        # Создаем и выполняем все задачи параллельно
        tasks = [asyncio.create_task(process_single_task(task)) for task in state["tasks"]]
        state["tasks_results"] = await asyncio.gather(*tasks, return_exceptions=True)
        
        # Обрабатываем исключения
        state["tasks_results"] = [
            f"Ошибка при обработке: {str(result)}" if isinstance(result, Exception) else result
            for result in state["tasks_results"]
        ]
        
        return state
   

    async def summarize_responses(self, state: AgentState) -> AgentState:
        """Суммаризирует ответы в финальный результат."""
        if not state.get("tasks_results") or not state.get("tasks"):
            state["error"] = "No responses to summarize"
            return state
            
        user_history = await asyncio.to_thread(
            self._get_user_history_text_sync,  # функция
            state["user_id"]                   # аргумент
        )
        prompt = ChatPromptTemplate.from_template(self.prompts["collect_final"])
        chain = prompt | self.llm_hard
        
        # Форматируем ответы для суммаризации
        task_responses_text = "\n\n".join([
            f"Вопрос: {task}\nОтвет: {answer}" 
            for task, answer in zip(state["tasks"], state["tasks_results"])
        ])
        
        original_question = state['user_input']
        
        response_obj = await chain.ainvoke({
            "user_history":user_history,
            "original_question": original_question,
            "task_responses": task_responses_text
        })
        
        state["model_output"] = response_obj.content.strip()
        return state

    async def validate_output(self, state: AgentState) -> AgentState:
        """Проверяем, что финальный ответ модели валиден."""
        prompt = ChatPromptTemplate.from_template(self.prompts["validate"])  # Можно использовать тот же промпт или создать отдельный
        chain = prompt | self.llm_easy
        
        # Проверяем сгенерированный ответ
        response_obj = await chain.ainvoke({"text": state["model_output"]})
        response = response_obj.content.strip().lower()
        
        
        if "valid" not in response:
            # Можно добавить fallback логику здесь
            state["error"] = "Generated response failed validation"
            state["is_valid_output"] = False
        else:
            state["is_valid_output"] = True
        
        return state
    
    async def handle_error(self, state: AgentState) -> AgentState:
        """Обрабатываем ошибки валидации."""
        state["model_output"] = "Извините, произошла ошибка при генерации ответа. Пожалуйста, попробуйте переформулировать вопрос."
        return state

In [9]:
agent = ProfkomAgent(llm_easy=llm_yandex_light,
                     llm_hard=llm_gpt,
                     redis_cache=SemanticRedisCache(db=0),
                     chat_manager=ChatHistoryManager(),
                     chroma_adapter=ChromaAdapter(text_type="query"),
                     text_embeder=MyEmbeddingFunction(text_type="text")
                    )

/var/folders/bt/mkhxym9x3dn9k9jn5cvtqbb00000gp/T/ipykernel_48105/954169021.py:36: DeprecationWarning: The class MyEmbeddingFunction does not implement __init__. This will be required in a future version.
  super().__init__(*args, **kwargs)


In [10]:
builder = StateGraph(AgentState, is_async=True)

# Добавляем все узлы
builder.add_node("check_redis_cache", agent.check_redis_cache)
builder.add_node("validate_input", agent.validate_text)
builder.add_node("decomposer", agent.decomposer)
builder.add_node("process_tasks", agent.process_all_tasks)
builder.add_node("summarize_responses", agent.summarize_responses)
builder.add_node("validate_output", agent.validate_output)  # НОВЫЙ узел для валидации ответа

# Устанавливаем начальную точку
builder.set_entry_point("check_redis_cache")

# Маршрутизация после проверки кэша
def route_after_cache(state: AgentState):
    return END if state["is_cached"] else "validate_input"

builder.add_conditional_edges("check_redis_cache", route_after_cache, {
    "validate_input": "validate_input",
    END: END
})

# Маршрутизация после валидации ввода
def route_after_validate_input(state: AgentState):
    return "decomposer" if state["is_valid_input"] else END

builder.add_conditional_edges("validate_input", route_after_validate_input, {
    "decomposer": "decomposer", 
    END: END
})

# Основная цепочка обработки
builder.add_edge("decomposer", "process_tasks")
builder.add_edge("process_tasks", "summarize_responses")
builder.add_edge("summarize_responses", "validate_output")

# Маршрутизация после валидации вывода
def route_after_validate_output(state: AgentState):
    return END if state["is_valid_output"] else "error_handler"

builder.add_conditional_edges("validate_output", route_after_validate_output, {
    END: END,
    "error_handler": "error_handler"
})

# Узел для обработки ошибок (опционально)
builder.add_node("error_handler", agent.handle_error)
builder.add_edge("error_handler", END)

graph = builder.compile()

In [21]:
graph

KeyboardInterrupt: 

# Text pipeline

In [22]:
user_info="Член профсоюзной организации: ППО Невинномысский Азот"
user_input=f"[{user_info}] | Приветствую! Имеет ли право работодатель вызывать меня в выходной день из-за остановки цеха?"

In [23]:
start_state = AgentState(
    user_id="141",
    user_input=user_input
)

In [24]:
result = await graph.ainvoke(start_state)

In [25]:
result

{'embedding_input': [array([ 2.76336670e-02, -8.69750977e-04, -4.71801758e-02, -3.04260254e-02,
         -2.07366943e-02, -5.50537109e-02,  1.72271729e-02, -4.02221680e-02,
          5.29174805e-02, -5.73425293e-02,  4.18090820e-02, -7.57446289e-02,
         -4.68444824e-02, -3.94821167e-03,  4.68750000e-02,  7.66372681e-03,
          7.37304688e-02, -4.29992676e-02, -7.28759766e-02, -2.27813721e-02,
         -4.88586426e-02,  8.70971680e-02,  8.59375000e-02, -6.19201660e-02,
         -8.61816406e-02, -2.12402344e-02,  5.68847656e-02, -2.67181396e-02,
          5.69763184e-02, -7.94677734e-02,  1.03149414e-01,  3.17993164e-02,
         -3.54614258e-02, -1.10961914e-01,  2.18811035e-02, -4.41894531e-02,
         -7.26699829e-03, -2.89764404e-02,  1.24511719e-02, -5.41076660e-02,
         -1.07055664e-01, -7.51342773e-02, -2.43530273e-02, -1.66320801e-02,
          7.03125000e-02, -7.24411011e-03, -8.49609375e-02, -2.82897949e-02,
          8.28857422e-02,  5.57250977e-02,  6.68945312e-0

In [26]:
len(result["tasks"])

3

In [27]:
result["tasks"]

['Имеет ли работодатель право вызывать сотрудника в выходной день из‑за остановки цеха?',
 'Какие нормативные акты регулируют вызов работников в выходные при непредвиденных обстоятельствах?',
 'Какова компенсация или порядок оплаты за работу в выходной день, если работник был вызван?']

In [28]:
print(result["model_output"])

Да, работодатель может вызвать вас в выходной, если цех остановлен из‑за непредвиденных обстоятельств.  
Он обязан оформить такой вызов письменно (смс, электронное письмо или приказ).  
Сначала уточните у руководства, почему нужен ваш выход и какие работы планируются.  
Проверьте ваш коллективный договор – в нём обычно прописано, как оплачивается работа в выходные.  
Если в договоре указано двойное вознаграждение, потребуйте его за отработанные часы.  
Если двойная оплата не предусмотрена, попросите предоставить вам другой свободный день с сохранением средней зарплаты.  
Откажитесь от работы без согласования оплаты – это ваше право.  
Если работодатель отказывается платить, обратитесь в профком ППО «Невинномысский Азот» за поддержкой.  
Профком поможет оформить жалобу в трудовую инспекцию или в суд.  
Сохраняйте все подтверждающие документы (вызов, ответы, расчёт оплаты) – они понадобятся при споре.


## Проверяем простой текст

In [18]:
user_info="Член профсоюзной организации: ППО Невинномысский Азот"
user_input=f"{user_info}] | Какой коллективный договор действует на предприятии? Имеет ли право работодатель изменить его самостоятельно?"

In [19]:
start_state = AgentState(
    user_id="141",
    user_input=user_input
)

In [20]:
result = await graph.ainvoke(start_state)

In [21]:
print(result["model_output"])

На предприятии О А «Невинномысский Азот» действует коллективный договор, который вступил в силу 1 января 2024 года и будет действовать до 31 декабря 2026 года.  

Работодатель не может менять условия договора сам, без согласия работников.  

Если вам предложили изменить что‑то в договоре, попросите написать об этом в письменной форме и передать в Комиссию по коллективным договорам (это может быть ваш профком).  

Комиссия обязана рассмотреть предложение в течение 14 дней и провести переговоры с работодателем.  

Все изменения фиксируются в дополнительном соглашении, которое подписывают обе стороны.  

Если предлагаются изменения, ухудшающие ваши условия (например, снижение зарплаты или льгот), их можно одобрить только на собрании трудового коллектива.  

Если вы заметили, что договор меняют без вашего согласия, сразу сообщите в профком — они подадут официальную жалобу работодателю.  

Для контроля держите копию текущего договора под рукой и проверяйте, совпадают ли новые условия с тем,

In [24]:
result["tasks"]

['Какой коллективный договор действует на предприятии ППО Невинномысский Азот?',
 'Имеет ли право работодатель изменить коллективный договор самостоятельно и какие процедуры при этом предусмотрены?']

In [23]:
result

{'embedding_input': [array([ 0.02218628, -0.04046631, -0.03100586, -0.04321289, -0.02449036,
         -0.0668335 ,  0.01686096, -0.0340271 ,  0.04492188,  0.0144577 ,
         -0.00893402, -0.02870178, -0.12493896,  0.02003479,  0.00615311,
         -0.01490021,  0.0682373 , -0.03018188, -0.05725098, -0.00314522,
         -0.03689575,  0.02503967,  0.01060486, -0.0447998 , -0.04257202,
         -0.04971313,  0.06311035, -0.04730225,  0.08544922, -0.06051636,
          0.10601807,  0.0166626 , -0.0177002 , -0.05575562, -0.02796936,
          0.04510498, -0.00317192, -0.05831909,  0.08300781, -0.01968384,
         -0.10699463, -0.08911133, -0.12158203, -0.01649475,  0.03918457,
          0.01072693, -0.0803833 , -0.05990601,  0.1005249 ,  0.05276489,
          0.03820801,  0.07757568, -0.0411377 ,  0.0647583 , -0.00017655,
          0.02397156, -0.06140137, -0.04351807,  0.00693893, -0.03649902,
          0.03610229, -0.0401001 , -0.05444336,  0.0418396 , -0.02442932,
          0.0597839